In [1]:
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_dataset

client = chromadb.PersistentClient(path="chroma_store")   # the notebook runs inside day07
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
col = client.get_or_create_collection("imdb", embedding_function=ef)

df = load_dataset("stanfordnlp/imdb", split="train").shuffle(seed=42).select(range(5000)).to_pandas()
col.add(
    ids=[f"doc-{i}" for i in range(len(df))],
    documents=df["text"].tolist(),
    metadatas=[{"sentiment": "positive" if l == 1 else "negative"} for l in df["label"]],
)
print("Stored:", col.count())

c:\Users\a4saa\ai-trainee\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
c:\Users\a4saa\ai-trainee\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Stored: 5000


In [2]:
res = col.query(
    query_texts=["a film with breathtaking cinematography"],
    n_results=5,
    where={"sentiment": "positive"},
)
for doc, dist in zip(res["documents"][0], res["distances"][0]):
    print(f"{dist:.3f}  {doc[:120]}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


0.868  The beginning of this film is a little clunky and also confusing , but sit tight, because you are in for the ride of you
0.876  This is the definite Lars von Trier Movie, my favorite, I rank it higher than "Breaking the waves" or the latest "Dancer
0.876  What's the best way to start a review of a movie like Der Todesking? Let me start by saying I've just come direct from v
0.878  Uggh! I really wasn't that impressed by this film, though I must admit that it is technically well made. It does get a 7
0.885  This is one of the best movies I've ever seen. It has very good acting by Hanks, Newman, and everyone else. Definitely J
